In [1]:
import pandas as pd
df = pd.read_csv('dataset_mood_smartphone.csv')

In [ ]:
df = pd.read_csv("data1C/data_per_day.csv")

feature_cols = df.columns.drop(["id", "date", "mood"])

df["num_nonzero"] = ((df[feature_cols] != 0) & (~df[feature_cols].isna())).sum(axis=1)

# delete rows where num_nonzero < 4 and mood is missing
df["is_fake"] = (df["num_nonzero"] < 4) & (df["mood"].isna())

df_clean = df[~df["is_fake"]].copy()

In [ ]:
df_clean["date"] = pd.to_datetime(df_clean["date"])
df_clean = df_clean.sort_values(["id", "date"])

# Create target variable for next day's mood
df_clean["mood_next_day"] = df_clean.groupby("id")["mood"].shift(-1)
df_clean["next_date"] = df_clean.groupby("id")["date"].shift(-1)

# Keep only rows where mood_next_day is not null and next_date is exactly one day after date
df_model = df_clean[
    (df_clean["mood_next_day"].notna()) &
    ((df_clean["next_date"] - df_clean["date"]).dt.days == 1)
].copy()

In [19]:
print("Original:", len(df))
print("After cleaning:", len(df_model))

Original: 1973
After cleaning: 1242


In [20]:
df_model[["date", "mood", "mood_next_day"]].head(10)

,date,mood,mood_next_day
7,2014-02-26,6.25,6.333333
25,2014-03-20,NaN,6.200000
26,2014-03-21,6.20,6.400000
27,2014-03-22,6.40,6.800000
28,2014-03-23,6.80,6.000000
29,2014-03-24,6.00,6.750000
30,2014-03-25,6.75,6.600000
31,2014-03-26,6.60,7.000000
32,2014-03-27,7.00,6.400000
33,2014-03-28,6.40,8.000000
